# NBA Insights

In [ ]:
import sqlite3
from pathlib import Path

from nba import visualisation, wrangling

DATA_PATH = Path.cwd() / "data" / "nba.sqlite"
con = sqlite3.connect(DATA_PATH)

## Finding 1: Home advantage comes from the fans, just not the way you'd think (Yvette)

Home teams win more often in almost every sport, and the NBA is no exception. But where does the edge come from? Using games going back to 1946, we look at whether it is shrinking, what happens when the fans are taken away, and how the crowd actually helps.

In [ ]:
print(wrangling.HOME_WIN_BY_SEASON_QUERY)
home_by_season = wrangling.home_win_by_season(con)
home_by_season.round(1)

In [ ]:
fig = visualisation.plot_home_advantage_trend(home_by_season)

**Home advantage has been shrinking for decades.** In the 1950s home teams won roughly three games in four. By 2022-23 that was down to 58%. Over the same period, average crowds about tripled, from around 6,000 to 18,000. So bigger crowds have not made home court stronger. The long-run decline probably has other causes, such as easier travel and more similar arenas, which this data can't test.

In [ ]:
print(wrangling.HOME_WIN_BY_SETTING_QUERY)
home_by_setting = wrangling.home_win_by_setting(con)
home_by_setting.round(1)

In [ ]:
fig = visualisation.plot_home_win_by_setting(home_by_setting)

**But take the fans away and the edge is gone.** Because of COVID, most 2020-21 games were played in empty arenas. Home teams won 60.4% of games with fans and just 50.4% without them, which is basically a coin flip. All-Star games, where "home" is only a label, also sit close to 50%. Preseason games don't count for anything, yet home teams win almost as often as in the regular season (58.1% vs 58.9% since 2000), so the edge doesn't seem to come from home teams trying harder. The playoffs are highest at 62.4%, but there the higher seed gets home court, so part of that is simply the better team playing at home.

In [ ]:
print(wrangling.CROWD_EFFECTS_QUERY)
crowd = wrangling.crowd_effects(con)
crowd.round(2)

In [ ]:
fig = visualisation.plot_crowd_effects(crowd)

**So how does the crowd do it?** Not by putting off the away team's shooters: they made 77.6% of their free throws with fans and 77.7% without. What changed was how many free throws each side got. With fans, home teams took 1.25 more free throws per game than their opponents; in empty arenas that fell to 0.50, and away teams were also called for fewer extra fouls (0.59 vs 0.16). That points to referees reacting to the crowd. One caveat: teams that are ahead late in games get extra free throws when the other side fouls on purpose, so some of this drop may just be because home teams won less.

**Is it really the fans?** Arenas reopened at different times, so the games with fans might just have involved stronger home teams. To check, each team is compared with itself: for the 26 teams that played home games both with and without fans, the average gap is 10.6 points, about the same as the overall gap of 10.

In [ ]:
print(wrangling.SAME_TEAM_CROWD_GAP_QUERY)
wrangling.same_team_crowd_gap(con).round(1)

## Finding 2: The three-pointer went from desperation shot to winning weapon (Yvette)

These days it can feel like every other shot is a three. Complete three-point numbers start in 1985-86, so we can follow how the shot took over the league, and whether shooting more of them actually wins games.

In [ ]:
print(wrangling.THREES_BY_SEASON_QUERY)
threes = wrangling.threes_by_season(con)
threes.round(1)

In [ ]:
fig = visualisation.plot_three_point_trend(threes)

**Threes now make up almost 40% of all shots, up from 4% in 1985-86.** You can even spot the three seasons (1994-97) when the NBA moved the line closer: attempts jumped, then dropped back when it moved out again. Accuracy has barely changed since the mid-1990s, at around 35-36%, so players are taking far more threes without making fewer of them. Playoff teams lean on the three even more than in the regular season, although they make a slightly lower share of them.

In [ ]:
print(wrangling.THREES_WIN_RATES_QUERY)
threes_wins = wrangling.threes_win_rates(con)
threes_wins.round(1)

In [ ]:
fig = visualisation.plot_three_point_win_rates(threes_wins)

**Making threes wins games. Just taking them doesn't.** In 1985-86 the team that took more threes lost 7 games out of 10. Back then the three was mostly a desperation shot for teams trying to catch up, so taking lots of them was a sign you were losing, not the reason why. Today the team that makes more threes wins about two games in three (66.9% in 2022-23), while the team that simply takes more is still close to a coin flip (51.9%). Making more threes also means scoring more points, so the interesting part is how much more it matters now than it used to.

### Data notes (Findings 1 and 2)
- All-Star games are stored under both "All Star" and "All-Star", and most appear twice, so they are de-duplicated by date and teams (72 games, 1951-2023; there was no game in 1999 because of the lockout).
- `game_info` has a few duplicated games, so it is grouped by `game_id` before joining.
- In 2020-21, a game with no attendance record (or 0) is treated as an empty arena. The share of such games falls month by month as arenas reopened.
- Attendance is only recorded for most games from 1962-63, and three-point data is only complete from 1985-86. A few seasons are missing from the data (for example 2012-13), and the lines simply join across them.
- The pre-commit hook strips outputs, so run the notebook from top to bottom to see the tables and plots.

## Finding 3: Is a fourth-quarter lead safe? Less than it used to be (Bryan)

Every fan has been tempted to switch off a game at the end of the third quarter. Should they? Using 53,568 regular season and playoff games since 1946 with quarter-by-quarter scores, we look at how safe a lead really is, whether that has changed, and which comebacks were the biggest of all.

In [ ]:
print(wrangling.LEAD_SAFETY_QUERY)
safety = wrangling.lead_safety(con)
safety.round(1)

In [ ]:
fig = visualisation.plot_lead_safety(safety)

**Stay tuned for a close game, but a blowout really is over.** A team leading by 1-5 points after three quarters wins only 63.7% of the time, so more than one game in three still turns around. At 11-15 points the leader wins 94.2% of the time, and at 21 or more it wins 99.8%: just 9 of 5,712 such games were lost.

In [ ]:
print(wrangling.BLOWN_LEADS_BY_DECADE_QUERY)
blown = wrangling.blown_leads_by_decade(con)
blown.round(2)

In [ ]:
fig = visualisation.plot_blown_leads_by_decade(blown)

**Big leads are getting less safe.** Teams leading by 10 or more after three quarters lost 3.4% of those games in the 1980s and 5.1% in the 2020s, and the rate has risen every decade in between. It is still rare, but about one and a half times as common. One possible reason is the three-pointer (Finding 2), which lets a trailing team catch up faster, but this data only shows that the two trends happen at the same time. The early decades are based on fewer games, so small differences there mean little.

In [ ]:
print(wrangling.BIGGEST_COMEBACKS_QUERY)
wrangling.biggest_comebacks(con)

**The greatest comebacks.** Milwaukee tops the list: 28 points down in Atlanta after three quarters in November 1977, and they won 117-115. Half of the top ten have come since 2012, and two were in the playoffs: the Clippers over Memphis in 2012 and Golden State over New Orleans in 2015. The list ranks the deficit at the end of the third quarter, so a team that was further behind earlier in a game is ranked only by where it stood at that point.

## Finding 4: How the NBA went from an American league to a global one (Bryan)

Some of the league's biggest stars today, like Nikola Jokic, Giannis Antetokounmpo and Luka Doncic, come from abroad, but it wasn't always so. Using the country listed for each player and the decade they joined, we follow how the NBA opened up to the world and where its international players come from.

In [ ]:
print(wrangling.INTERNATIONAL_SHARE_QUERY)
international = wrangling.international_share(con)
international.round(1)

In [ ]:
fig = visualisation.plot_international_share(international)

**The league opened up within a generation.** In this data, only one player from outside the USA joined in the NBA's first 39 years: Norm Baker from Canada in 1946. The share then rose to 1.7% in the 1980s, 6.3% in the 1990s and 20.5% in the 2000s. It peaked at 23.8% in the 2010s, when about one new player in four came from abroad and new players came from 51 countries, including the USA. The 2020s figure (20.7%) only covers 2020 to 2022, so it is too early to say the trend has turned.

In [ ]:
print(wrangling.TOP_COUNTRIES_QUERY)
countries = wrangling.top_countries(con)
countries

In [ ]:
fig = visualisation.plot_top_countries(countries)

**Canada leads, but France has caught up fast.** Canada has the most players outside the USA with 35, and France is close behind with 32, even though its first player in the data only joined in 2000. Serbia and Croatia come next with 21 and 18, both with their first players arriving in 1989.

### Data notes (Findings 3 and 4)
- `line_score` and `game` disagree on which team was at home in about 47% of games. Finding 3 therefore ignores home and away, takes all scores from `line_score`, and uses `game` only for dates and season types.
- Some old games have mistyped quarter scores. For example, the Lakers have 3 points in Q3 on 1964-01-15, which would be a fake 29-point comeback. Games without overtime whose quarters do not add up to the final score are left out (66 games, mostly before 1970).
- Quarter scores are missing for most games in the 1940s and some in the 1950s. Games tied after three quarters are left out, and the 40 games stored twice in `line_score` are counted once.
- `common_player_info` has 3,632 of the 4,815 players in `player`. Some big names are missing, such as LeBron James, so player counts are too low.
- `country` is not always nationality: Kyrie Irving is listed as Australia and Tim Duncan as US Virgin Islands. Anything other than `USA` counts as international, including Puerto Rico and the US Virgin Islands.
- The pre-commit hook strips outputs, so run the notebook from top to bottom to see the tables and plots.